# V6.5-17 — Data v3 (6 class, box đã thắt) + YOLO26m teacher → distill YOLO26n (Raspberry Pi)
Mục tiêu: **mAP50 MEAN_5 > 0.82** (5 class lỗi) trên `val_clean`, model chạy được trên Raspberry Pi 3 (CPU, NCNN).

- **Dataset** `datasets/v6p5_6cls_v3` = **bộ chung** cho mọi model (YOLO, RT-DETR, StructDETR):
  v2 + giữ lại insulator + rule H (thắt box insulator / bird_nest / FO bằng SAM, chỉ thu nhỏ).
  YOLO đọc `labels/`, RT-DETR/DETR đọc `annotations/instances_*.json` (cùng label, xem `labels_sha256` trong `manifest.json`).
- **insulator chỉ được gán nhãn ở ảnh chụp xa** (ảnh cận cảnh sứ vỡ không có box insulator) → AP insulator không phản ánh đúng,
  thước đo chính là **MEAN_5** (5 class lỗi), insulator báo cáo riêng.
- **Val**: `val` = VAL gốc (frozen, so được với run cũ); `val_clean` = đã sửa nhãn + thắt box → thước đo chính.
- **Teacher** khởi tạo từ teacher v2 (5 class, 80 epoch) rồi fine-tune 40 epoch sang 6 class (head class khởi tạo lại).
- **Thứ tự**: cell 3 teacher → 3b so với V6.4 → 4 student distill → 5 export NCNN → 6 eval.
- Mỗi cell train **tự resume** nếu bị ngắt, bỏ qua nếu đã xong; **watchdog** 15 phút không có batch → tự ngắt + resume (tối đa 3 lần).
- NCNN làm mAP50 giảm ~1.3 điểm → student `.pt` cần ≥ ~0.83.


In [1]:
# 1. CONFIG + GATE (không train)
from pathlib import Path
import os, csv, time, logging
import torch, yaml, ultralytics
from ultralytics import YOLO
from ultralytics.utils import LOGGER

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next(p for p in candidates if (p / "pyproject.toml").is_file() and (p / "legacy/notebooks/v6p5").is_dir())
WEIGHTS = PROJECT / "legacy/notebooks/v6p5/weights"
DATASET = PROJECT.parent / "datasets/v6p5_6cls_v3"
DATA_YAML, VAL_CLEAN_YAML = DATASET / "data.yaml", DATASET / "data_val_clean.yaml"
OUT = PROJECT / "artifacts/v6p5_v3"
TEACHER_V2 = PROJECT / "artifacts/v6p5_v2/runs/teacher/weights/best.pt"  # 5 class, khởi tạo cho teacher v3
RUNS = OUT / "runs"
NAMES = ["insulator", "insulator_broken", "insulator_flashover", "bird_nest", "foreign_object", "broken_strand"]
DEFECTS = NAMES[1:]  # MEAN_5 = thước đo chính
TARGET_MAP50 = 0.82
IMGSZ, DEVICE, SEED = 640, 0, 20261012
WORKERS = min(8, os.cpu_count() or 4)
os.environ.setdefault("PYTHONHASHSEED", str(SEED))

# Công thức chung cho teacher và student (AdamW + cosine; plateau ở v6p5-14 giảm LR quá muộn)
COMMON = dict(data=str(DATA_YAML), imgsz=IMGSZ, device=DEVICE, workers=WORKERS, seed=SEED, deterministic=True,
              optimizer="AdamW", lr0=1e-3, lrf=0.05, cos_lr=True, weight_decay=1e-3, warmup_epochs=3.0,
              close_mosaic=15, cache="disk", amp=True, pretrained=True, plots=False, save_period=5)

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU — chuyển studio sang GPU rồi chạy lại.")
if ultralytics.__version__ != "8.4.172":
    raise RuntimeError(f"Cần ultralytics==8.4.172 (có distill_model); hiện tại={ultralytics.__version__}")
import ncnn  # export cho Raspberry Pi; thiếu thì: pip install ncnn pnnx

cfg = yaml.safe_load(DATA_YAML.read_text())
assert [cfg["names"][k] for k in sorted(cfg["names"])] == NAMES
for split in ["images/train", "images/val", "images/val_clean"]:
    print(split, sum(1 for _ in (DATASET / split).iterdir()))
for w in ["yolo26m.pt", "yolo26n.pt"]:
    assert (WEIGHTS / w).is_file(), w
assert TEACHER_V2.is_file(), TEACHER_V2
assert "H_tighten" in yaml.safe_load((DATASET / "manifest.json").read_text())["val_policy"], "dataset chưa thắt box"
OUT.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0), "| Ultralytics:", ultralytics.__version__, "| workers:", WORKERS)

images/train 17903
images/val 2272
images/val_clean 2271
GPU: NVIDIA L4 | Ultralytics: 8.4.172 | workers: 8


In [2]:
# 2. HÀM TRAIN: resume / bỏ qua nếu xong / watchdog chống treo / tự thử lại, in 1 dòng mỗi epoch
import gc, shutil, signal, threading
def _epoch_line(trainer):
    m, loss = trainer.metrics, trainer.label_loss_items(trainer.tloss)
    # sau epoch cuối ultralytics val lại bằng best.pt và gọi lại callback này
    final = trainer.stop and getattr(trainer, "_stop_seen", False)
    trainer._stop_seen = trainer.stop
    tag = "best.pt (val cuối)" if final else f"ep {trainer.epoch + 1:>3}/{trainer.epochs}"
    print(f"{trainer.args.name} {tag}  "
          f"box {float(loss['train/box_loss']):.3f} cls {float(loss['train/cls_loss']):.3f}  "
          f"mAP50 {m.get('metrics/mAP50(B)', 0):.4f}  "
          f"mAP50-95 {m.get('metrics/mAP50-95(B)', 0):.4f}  lr {trainer.optimizer.param_groups[0]['lr']:.2e}  "
          f"{time.strftime('%H:%M')}", flush=True)

HANG_MINUTES = 15  # không có batch nào chạy xong trong khoảng này → coi là treo
MAX_RETRIES = 3

class Watchdog:
    """Gửi SIGINT cho kernel nếu train không còn tiến triển (giống bấm Interrupt) để cell tự resume."""
    def __init__(self, minutes):
        self.limit, self.beat, self.armed, self.fired = minutes * 60, time.time(), False, False
        self._stop = threading.Event()
    def heartbeat(self, *_):  # gọi sau mỗi batch train/val
        self.beat, self.armed = time.time(), True
    def _run(self):
        while not self._stop.wait(30):
            if self.armed and time.time() - self.beat > self.limit:
                self.fired = True
                print(f"[watchdog] {self.limit // 60} phút không có batch nào chạy xong → ngắt để resume", flush=True)
                os.kill(os.getpid(), signal.SIGINT)
                return
    def __enter__(self):
        threading.Thread(target=self._run, daemon=True).start()
        return self
    def __exit__(self, *exc):
        self._stop.set()

def _latest_checkpoint(run):
    """last.pt, nếu hỏng (bị kill lúc đang ghi) thì lùi về epochN.pt mới nhất."""
    epochs = sorted(run.glob("weights/epoch*.pt"), key=lambda p: -int(p.stem[5:]))
    for p in [run / "weights/last.pt", *epochs]:
        if p.is_file():
            try:
                return p, torch.load(p, map_location="cpu", weights_only=False)
            except Exception as e:
                print(f"[{run.name}] checkpoint hỏng, bỏ qua {p.name}: {e}")
    return None, None

def train_stage(name, model, **kw):
    run = RUNS / name
    best = run / "weights/best.pt"
    for attempt in range(1, MAX_RETRIES + 2):
        ckpt, ck = _latest_checkpoint(run)
        done = ck is not None and (ck.get("epoch", -1) == -1  # ultralytics đánh dấu run đã xong bằng epoch=-1
                                   or ck["epoch"] + 1 >= ck.get("train_args", {}).get("epochs", 10**9))
        if done and (ck.get("train_args", {}).get("epochs") == kw.get("epochs", ck.get("train_args", {}).get("epochs"))
                     or ck.get("epoch", -1) == -1):
            print(f"[{name}] đã xong, bỏ qua: {best}")
            return best
        if ck is not None and "epochs" in kw and ck.get("train_args", {}).get("epochs") != kw["epochs"]:
            print(f"[{name}] run cũ train {ck['train_args']['epochs']} epoch ≠ {kw['epochs']} → bắt đầu lại")
            ck = None
        if ck is not None:
            print(f"[{name}] resume từ {ckpt.name} (đã xong {ck['epoch'] + 1} epoch)")
            yolo, args = YOLO(str(ckpt)), dict(resume=True)
        else:
            if run.is_dir():  # không có checkpoint dùng được → làm lại từ đầu
                shutil.rmtree(run)
            print(f"[{name}] bắt đầu: {model}")
            yolo, args = YOLO(str(model)), dict(project=str(RUNS), name=name, exist_ok=False, **COMMON, **kw)
        wd = Watchdog(HANG_MINUTES)
        for event in ("on_train_batch_end", "on_val_batch_end", "on_fit_epoch_end"):
            yolo.add_callback(event, wd.heartbeat)
        yolo.add_callback("on_fit_epoch_end", _epoch_line)
        level = LOGGER.level
        LOGGER.setLevel(logging.WARNING)  # tắt progress bar + log chi tiết → output không bị treo
        try:
            with wd:
                yolo.train(**args)
            print(f"[{name}] xong: {best}")
            return best
        except KeyboardInterrupt:
            if not wd.fired:
                print(f"[{name}] dừng theo yêu cầu. Chạy lại cell này để resume.")
                raise
            print(f"[{name}] treo → resume (lần {attempt}/{MAX_RETRIES})")
        except Exception as e:
            print(f"[{name}] lỗi {type(e).__name__}: {e} → resume (lần {attempt}/{MAX_RETRIES})")
        finally:
            LOGGER.setLevel(level)
            del yolo
            gc.collect()
            torch.cuda.empty_cache()
        time.sleep(30)
    raise RuntimeError(f"[{name}] thất bại sau {MAX_RETRIES} lần thử lại — xem log ở trên")

In [3]:
# 3. TEACHER — YOLO26m khởi tạo từ teacher v2 (5 class), fine-tune 40 epoch sang 6 class (~4 giờ trên L4)
TEACHER_BEST = train_stage("teacher", TEACHER_V2, epochs=40, batch=24, patience=15)


[teacher] bắt đầu: /teamspace/studios/this_studio/Uav-Powerline-Defect-Detection/artifacts/v6p5_v2/runs/teacher/weights/best.pt
train: Caching images (45.9GB Disk): 100% ━━━━━━━━━━━━ 17903/17903 137.5it/s 2:100.1ss
val: Caching images (6.7GB Disk): 100% ━━━━━━━━━━━━ 2272/2272 160.1it/s 14.2s0.0s
teacher ep   1/40  box 1.772 cls 2.087  mAP50 0.6724  mAP50-95 0.3226  lr 3.33e-04  03:40
teacher ep   2/40  box 1.645 cls 1.122  mAP50 0.7280  mAP50-95 0.3594  lr 6.65e-04  03:46
teacher ep   3/40  box 1.628 cls 1.099  mAP50 0.7506  mAP50-95 0.3761  lr 9.94e-04  03:52
teacher ep   4/40  box 1.597 cls 1.057  mAP50 0.7540  mAP50-95 0.3710  lr 9.87e-04  03:58
teacher ep   5/40  box 1.583 cls 1.046  mAP50 0.7320  mAP50-95 0.3620  lr 9.77e-04  04:03
teacher ep   6/40  box 1.577 cls 1.018  mAP50 0.7467  mAP50-95 0.3752  lr 9.64e-04  04:09
teacher ep   7/40  box 1.561 cls 0.995  mAP50 0.7544  mAP50-95 0.3786  lr 9.48e-04  04:15
teacher ep   8/40  box 1.547 cls 0.984  mAP50 0.7606  mAP50-95 0.3804  lr

In [4]:
# 3b. SO SÁNH VỚI V6.4-clean YOLO11m@1024 (baseline cũ, tải từ Camber) — ~3 phút GPU
import pandas as pd

V64 = PROJECT / "artifacts/v6p4_clean"  # best.pt 6 class, cùng thứ tự class với v3
CMP = [  # (tên, weights, imgsz, data val gốc, data val_clean)
    ("V6.4 YOLO11m@1024", V64 / "best.pt", 1024, DATA_YAML, VAL_CLEAN_YAML),
    ("Teacher YOLO26m@640", RUNS / "teacher/weights/best.pt", 640, DATA_YAML, VAL_CLEAN_YAML),
]
rows = []
for tag, w, sz, d_val, d_clean in CMP:
    for vname, vyaml in [("val", d_val), ("val_clean", d_clean)]:
        m = YOLO(str(w))
        b = m.val(data=str(vyaml), imgsz=sz, batch=8, device=DEVICE, plots=False, verbose=False,
                  project=str(OUT / "eval_cmp"), name=f"{tag.split()[0]}_{vname}", exist_ok=True).box
        for i, c in enumerate(b.ap_class_index):  # đủ 6 class; MEAN_5 tính riêng trên 5 class lỗi
            rows.append({"model": tag, "val": vname, "class": m.names[int(c)], "P": float(b.p[i]),
                         "R": float(b.r[i]), "AP50": float(b.ap50[i]), "AP50_95": float(b.ap[i])})
cmp = pd.DataFrame(rows)
mean5 = cmp[cmp["class"].isin(DEFECTS)].groupby(["model", "val"], as_index=False)[["P", "R", "AP50", "AP50_95"]].mean().assign(**{"class": "MEAN_5"})
cmp = pd.concat([cmp, mean5], ignore_index=True)
cmp.to_csv(OUT / "compare_v6p4.csv", index=False)
display(cmp.pivot_table(index=["val", "class"], columns="model", values=["P", "R", "AP50", "AP50_95"]).round(3))

Ultralytics 8.4.172 🚀 Python-3.14.7 torch-2.14.0+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
YOLO11m summary (fused): 125 layers, 20,034,658 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 289.4±276.4 MB/s, size: 242.4 KB)
val: Scanning /teamspace/studios/this_studio/datasets/v6p5_6cls_v3/labels/val.cache... 2272 images, 179 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2272/2272 733.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 284/284 3.7it/s 1:160.2sss
                   all       2272       2903      0.878       0.78      0.839      0.472
Speed: 1.4ms preprocess, 21.1ms inference, 0.0ms loss, 1.1ms postprocess per image
Ultralytics 8.4.172 🚀 Python-3.14.7 torch-2.14.0+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
YOLO11m summary (fused): 125 layers, 20,034,658 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 355.0±230.7 MB/s, size: 298.4 KB)
val: Scanning /

AP50                    \
model                         Teacher YOLO26m@640 V6.4 YOLO11m@1024   
val       class                                                       
val       MEAN_5                            0.768             0.811   
          bird_nest                         0.851             0.880   
          broken_strand                     0.779             0.788   
          foreign_object                    0.521             0.656   
          insulator                         0.949             0.977   
          insulator_broken                  0.870             0.873   
          insulator_flashover               0.820             0.858   
val_clean MEAN_5                            0.799             0.798   
          bird_nest                         0.923             0.942   
          broken_strand                     0.779             0.788   
          foreign_object                    0.606             0.530   
          insulator                         0.947             0.971   
          insulator_broken                  0.870             0.873   
          insulator_flashover               0.820             0.858   

                                          AP50_95                    \
model                         Teacher YOLO26m@640 V6.4 YOLO11m@1024   
val       class                                                       
val       MEAN_5                            0.389             0.424   
          bird_nest                         0.516             0.540   
          broken_strand                     0.354             0.368   
          foreign_object                    0.257             0.376   
          insulator                         0.654             0.709   
          insulator_broken                  0.457             0.460   
          insulator_flashover               0.362             0.378   
val_clean MEAN_5                            0.425             0.417   
          bird_nest                         0.568             0.587   
          broken_strand                     0.354             0.368   
          foreign_object                    0.385             0.291   
          insulator                         0.650             0.663   
          insulator_broken                  0.457             0.460   
          insulator_flashover               0.362             0.378   

                                                P                    \
model                         Teacher YOLO26m@640 V6.4 YOLO11m@1024   
val       class                                                       
val       MEAN_5                            0.846             0.865   
          bird_nest                         0.891             0.894   
          broken_strand                     0.846             0.824   
          foreign_object                    0.724             0.775   
          insulator                         0.938             0.942   
          insulator_broken                  0.915             0.924   
          insulator_flashover               0.854             0.907   
val_clean MEAN_5                            0.826             0.821   
          bird_nest                         0.922             0.926   
          broken_strand                     0.848             0.839   
          foreign_object                    0.593             0.505   
          insulator                         0.940             0.940   
          insulator_broken                  0.916             0.926   
          insulator_flashover               0.853             0.907   

                                                R                    
model                         Teacher YOLO26m@640 V6.4 YOLO11m@1024  
val       class                                                      
val       MEAN_5                            0.733             0.746  
          bird_nest                         0.875             0.883  
          broken_strand                     0.754             0.698  
 

In [ ]:
# 4. STUDENT — YOLO26n distill từ teacher (~5,5 giờ trên L4: ~5,4 phút/epoch vì chạy thêm teacher mỗi batch)
TEACHER_BEST = RUNS / "teacher/weights/best.pt"
assert TEACHER_BEST.is_file(), "Chạy cell 3 (teacher) trước"
STUDENT_BEST = train_stage("student_distill", WEIGHTS / "yolo26n.pt", epochs=60, batch=32, patience=15,
                           distill_model=str(TEACHER_BEST))

[student_distill] bắt đầu: /teamspace/studios/this_studio/Uav-Powerline-Defect-Detection/legacy/notebooks/v6p5/weights/yolo26n.pt
train: Caching images (45.9GB Disk): 100% ━━━━━━━━━━━━ 17903/17903 9.5Kit/s 1.9s<0.2s
val: Caching images (6.7GB Disk): 100% ━━━━━━━━━━━━ 2272/2272 10.4Kit/s 0.2s0ss
student_distill ep   1/60  box 2.137 cls 3.756  mAP50 0.2276  mAP50-95 0.0946  lr 3.33e-04  07:44
student_distill ep   2/60  box 1.977 cls 2.688  mAP50 0.3769  mAP50-95 0.1729  lr 6.66e-04  07:49
student_distill ep   3/60  box 1.918 cls 2.386  mAP50 0.4025  mAP50-95 0.1870  lr 9.97e-04  07:55
student_distill ep   4/60  box 1.912 cls 2.243  mAP50 0.4205  mAP50-95 0.2036  lr 9.94e-04  08:00
student_distill ep   5/60  box 1.870 cls 2.086  mAP50 0.4597  mAP50-95 0.2210  lr 9.90e-04  08:06
student_distill ep   6/60  box 1.846 cls 2.002  mAP50 0.4575  mAP50-95 0.2141  lr 9.84e-04  08:11
student_distill ep   7/60  box 1.839 cls 1.910  mAP50 0.4760  mAP50-95 0.2266  lr 9.77e-04  08:17
student_distill ep

In [ ]:
# 5. EXPORT NCNN cho Raspberry Pi
STUDENT_BEST = RUNS / "student_distill/weights/best.pt"
NCNN_DIR = STUDENT_BEST.with_name("best_ncnn_model")
if not NCNN_DIR.is_dir():
    YOLO(str(STUDENT_BEST)).export(format="ncnn", imgsz=IMGSZ, half=False)
print("Copy thư mục này sang Pi:", NCNN_DIR)
print("Trên Pi: pip install ultralytics ncnn; YOLO('best_ncnn_model', task='detect').predict('anh.jpg', imgsz=640)")

In [ ]:
# 6. EVAL trên val gốc + val_clean, kiểm tra mục tiêu
import pandas as pd

MODELS = {
    "teacher_y26m": RUNS / "teacher/weights/best.pt",
    "student_distill": RUNS / "student_distill/weights/best.pt",
    "student_distill_ncnn": RUNS / "student_distill/weights/best_ncnn_model",  # bản chạy trên Pi (CPU)
}
rows = []
for vname, vyaml in [("val", DATA_YAML), ("val_clean", VAL_CLEAN_YAML)]:
    for tag, w in MODELS.items():
        if not w.exists():
            continue
        ncnn_model = w.is_dir()
        b = YOLO(str(w), task="detect").val(data=str(vyaml), imgsz=IMGSZ, batch=1 if ncnn_model else 16,
                                            device="cpu" if ncnn_model else DEVICE, plots=False, verbose=False,
                                            project=str(OUT / "eval"), name=f"{tag}_{vname}", exist_ok=True).box
        rows.append({"model": tag, "val": vname, "class": "ALL_6", "P": float(b.mp), "R": float(b.mr), "AP50": float(b.map50), "AP50_95": float(b.map)})
        for i, (c, a50, a) in enumerate(zip(b.ap_class_index, b.ap50, b.ap)):
            rows.append({"model": tag, "val": vname, "class": NAMES[int(c)], "P": float(b.p[i]), "R": float(b.r[i]), "AP50": float(a50), "AP50_95": float(a)})
res = pd.DataFrame(rows)
mean5 = res[res["class"].isin(DEFECTS)].groupby(["model", "val"], as_index=False)[["P", "R", "AP50", "AP50_95"]].mean()
res = pd.concat([res, mean5.assign(**{"class": "MEAN_5"})], ignore_index=True)
res.to_csv(OUT / "eval.csv", index=False)
display(res.pivot_table(index=["val", "class"], columns="model", values=["P", "R", "AP50", "AP50_95"]).round(4))
pi = res.query("model == 'student_distill_ncnn' and val == 'val_clean' and `class` == 'MEAN_5'")
if len(pi):
    v = float(pi.AP50.iloc[0])
    print(f"Model Raspberry Pi (NCNN) trên val_clean: mAP50 MEAN_5 = {v:.4f} →",
          "ĐẠT" if v > TARGET_MAP50 else "CHƯA ĐẠT", f"mục tiêu {TARGET_MAP50}")

In [ ]:
# 7. XEM TIẾN ĐỘ từ results.csv (khi mở lại notebook / kernel đã restart; không ảnh hưởng train)
import datetime as dt
for stage in ["teacher", "student_distill"]:
    f = RUNS / stage / "results.csv"
    if not f.exists():
        continue
    rows_ = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(f.open())]
    best = max(rows_, key=lambda r: float(r["metrics/mAP50(B)"]))
    print(f"== {stage}: {len(rows_)} epoch | best mAP50 {float(best['metrics/mAP50(B)']):.4f} (ep {best['epoch']}) "
          f"| cập nhật {dt.datetime.fromtimestamp(f.stat().st_mtime):%H:%M:%S}")
    for r in rows_[-10:]:
        print(f"  ep {r['epoch']:>3}  box {float(r['train/box_loss']):.3f}  mAP50 {float(r['metrics/mAP50(B)']):.4f}  "
              f"mAP50-95 {float(r['metrics/mAP50-95(B)']):.4f}  lr {float(r['lr/pg0']):.2e}")